# Programming Assignment: Feature-Target Relationship Mapping


Welcome to the **Feature-Target Relationship Mapping** assignment!

This is the culminating EDA step — systematically exploring which features relate
to the target variable `returned`. This mapping directly informs feature selection
and model building in subsequent modules.

**Business Context:** An e-commerce retailer wants to predict whether an order will
be returned. Before training any model, you must understand *which* features carry
signal about returns.

**What you will build:**
- Pearson correlations between numeric features and the binary target
- A horizontal bar chart that makes those correlations immediately readable
- Group mean comparisons that reveal which regions have the highest return rates
- KDE overlays that show distributional differences between returned and not-returned orders
- A unified relationship summary table to prioritise features for modelling

> **Tip:** A small correlation doesn't mean a feature is useless — it just means the
> *linear* relationship is weak. Always check KDE overlaps too.


---
<a name='submission'></a>
<h4 style="color:green; font-weight:bold;">TIPS FOR SUCCESSFUL GRADING:</h4>

* Look for `### START CODE HERE ###` and `### END CODE HERE ###` markers
* Do not modify code outside these markers
* You can add experimental cells but they won't be graded

---


## Table of Contents
- [Imports](#imports)
- [1 - Dataset Overview](#1---dataset-overview)
- [2 - Feature-Target Correlations](#2---feature-target-correlations)
    - **[Exercise 1 - compute_feature_target_correlations](#exercise-1---compute_feature_target_correlations)**
- [3 - Correlation Bar Chart](#3---correlation-bar-chart)
    - **[Exercise 2 - plot_feature_target_correlations](#exercise-2---plot_feature_target_correlations)**
- [4 - Categorical Group Means](#4---categorical-group-means)
    - **[Exercise 3 - compute_group_means](#exercise-3---compute_group_means)**
- [5 - KDE Overlays by Target Class](#5---kde-overlays-by-target-class)
    - **[Exercise 4 - plot_kde_by_target](#exercise-4---plot_kde_by_target)**
- [6 - Relationship Summary Table](#6---relationship-summary-table)
    - **[Exercise 5 - build_relationship_summary](#exercise-5---build_relationship_summary)**
- [7 - Putting It All Together](#7---putting-it-all-together)


In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt


In [ ]:
import helper_utils
import unittests


<a name='1---dataset-overview'></a>
## 1 - Dataset Overview

We work with a simulated e-commerce orders dataset of **300 rows**, representing
individual purchase records.

| Column | Type | Description |
|--------|------|-------------|
| `order_value` | numeric | Total value of the order (USD) |
| `days_to_ship` | numeric | Days from purchase to shipment |
| `items_ordered` | numeric | Number of items in the order |
| `customer_age` | numeric | Age of the customer |
| `region` | categorical | Sales region (North, South, East, West, Central) |
| `returned` | **binary target** | 1 = returned, 0 = not returned |
| `days_since_last_purchase` | numeric | Days since the customer's previous purchase |

Numeric features we will analyse: `order_value`, `days_to_ship`, `items_ordered`,
`customer_age`, `days_since_last_purchase`.


In [ ]:
df = helper_utils.get_orders_df()
print("Shape:", df.shape)
print()
df.head()


<a name='2---feature-target-correlations'></a>
## 2 - Feature-Target Correlations

**Pearson Correlation** measures the linear relationship between a numeric feature
and the target. For a binary target (0/1) it is equivalent to the point-biserial
correlation. Values range from **-1** (perfect negative) to **+1** (perfect
positive), with **0** meaning no linear relationship.

<a name='exercise-1---compute_feature_target_correlations'></a>
### **Exercise 1 - `compute_feature_target_correlations`**

**Your Task:**
Compute the Pearson correlation of each numeric feature with the binary target
column, and return the results as a sorted `pd.Series`.

**Requirements:**
- Iterate over `numeric_cols` and compute `round(df[col].corr(df[target_col]), 4)`
  for each column.
- Return a `pd.Series` indexed by column name.
- Sort by **absolute value descending** (strongest relationship first).

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand)</font></b></summary>

  **Step-by-step guidance:**

  1. Build `correlations = {}` and fill it with `correlations[col] = round(df[col].corr(df[target_col]), 4)`.
  2. Convert to Series: `series = pd.Series(correlations)`.
  3. Sort by absolute value: `series.reindex(series.abs().sort_values(ascending=False).index)`.

</details>


In [ ]:
# GRADED FUNCTION: compute_feature_target_correlations
def compute_feature_target_correlations(df, numeric_cols, target_col):
    """
    Compute Pearson correlation of each numeric column with the target.

    Args:
        df (pd.DataFrame): Input DataFrame.
        numeric_cols (list): List of numeric column names.
        target_col (str): Name of the binary target column.

    Returns:
        pd.Series: Correlations indexed by column name, sorted by absolute
                   value descending. Values rounded to 4 decimal places.
    """
    ### START CODE HERE ###
    # YOUR CODE HERE
    raise NotImplementedError("Implement this function")
    ### END CODE HERE ###


In [ ]:
# Demo: compute correlations for all 5 numeric columns
numeric_cols = [
    "order_value", "days_to_ship", "items_ordered",
    "customer_age", "days_since_last_purchase"
]
correlations = compute_feature_target_correlations(df, numeric_cols, "returned")
print("Feature correlations with 'returned' (sorted by |r| descending):")
print(correlations)


In [ ]:
# Test your code!
unittests.exercise_1(compute_feature_target_correlations)


<a name='3---correlation-bar-chart'></a>
## 3 - Correlation Bar Chart

A **horizontal bar chart** makes it easy to compare correlation magnitudes at a
glance:
- Bars extending **right** (blue) → positive association with `returned = 1`.
- Bars extending **left** (red) → negative association.
- Bar length shows the strength of the relationship.

<a name='exercise-2---plot_feature_target_correlations'></a>
### **Exercise 2 - `plot_feature_target_correlations`**

**Your Task:**
Create a horizontal bar chart of the feature–target correlations.

**Requirements:**
- Call `compute_feature_target_correlations` to get the correlation Series.
- Sort the Series **ascending** before plotting (so the strongest bar appears at
  the top with `barh`).
- Colour bars: **blue** (`#4C72B0`) for positive values, **red** (`#C44E52`) for
  negative values.
- Set the title to `f"Feature Correlations with {target_col}"` and xlabel to
  `"Pearson Correlation"`.
- Add a vertical line at `x=0` (black, linewidth 0.8).
- Return the `Axes` object.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand)</font></b></summary>

  **Step-by-step guidance:**

  1. Call `compute_feature_target_correlations(df, numeric_cols, target_col)`.
  2. Sort ascending: `sorted_series = series.sort_values()`.
  3. Build colours: `['#4C72B0' if v >= 0 else '#C44E52' for v in sorted_series.values]`.
  4. `ax.barh(sorted_series.index, sorted_series.values, color=colors, edgecolor='white')`.
  5. `ax.axvline(0, color='black', linewidth=0.8)`.
  6. Set title and xlabel, then `plt.tight_layout()`.

</details>


In [ ]:
# GRADED FUNCTION: plot_feature_target_correlations
def plot_feature_target_correlations(df, numeric_cols, target_col, figsize=(8, 4)):
    """
    Plot a horizontal bar chart of feature-target Pearson correlations.

    Args:
        df (pd.DataFrame): Input DataFrame.
        numeric_cols (list): List of numeric column names.
        target_col (str): Name of the binary target column.
        figsize (tuple): Figure size. Default (8, 4).

    Returns:
        matplotlib.axes.Axes: The axes containing the bar chart.
    """
    ### START CODE HERE ###
    # YOUR CODE HERE
    raise NotImplementedError("Implement this function")
    ### END CODE HERE ###


In [ ]:
# Demo: correlation bar chart for all 5 numeric columns
numeric_cols = [
    "order_value", "days_to_ship", "items_ordered",
    "customer_age", "days_since_last_purchase"
]
ax = plot_feature_target_correlations(df, numeric_cols, "returned")
plt.show()


In [ ]:
# Test your code!
unittests.exercise_2(plot_feature_target_correlations)


<a name='4---categorical-group-means'></a>
## 4 - Categorical Group Means

For categorical features we cannot compute Pearson correlation directly. Instead,
we compute the **mean of the target within each category**.

For a binary target like `returned`, the group mean equals the **return rate** for
that category:

$$\text{return rate}_{\text{group}} = \frac{\text{# returned in group}}{\text{# total in group}}$$

A high return rate for a specific region means customers there are more likely to
return their orders — a valuable signal for the model.

<a name='exercise-3---compute_group_means'></a>
### **Exercise 3 - `compute_group_means`**

**Your Task:**
Compute the mean of the target column grouped by a categorical column, and return
the results sorted by mean value descending.

**Requirements:**
- Use `df.groupby(categorical_col)[target_col].mean()` to compute group means.
- Sort the result **descending** (highest return rate first).
- Return a `pd.Series` indexed by category names.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand)</font></b></summary>

  **Step-by-step guidance:**

  1. `means = df.groupby(categorical_col)[target_col].mean()`.
  2. Sort: `.sort_values(ascending=False)`.
  3. Return the sorted Series directly.

</details>


In [ ]:
# GRADED FUNCTION: compute_group_means
def compute_group_means(df, categorical_col, target_col):
    """
    Compute the mean of the target grouped by a categorical column.

    Args:
        df (pd.DataFrame): Input DataFrame.
        categorical_col (str): Name of the categorical column.
        target_col (str): Name of the binary target column.

    Returns:
        pd.Series: Mean target value per category, sorted descending.
    """
    ### START CODE HERE ###
    # YOUR CODE HERE
    raise NotImplementedError("Implement this function")
    ### END CODE HERE ###


In [ ]:
# Demo: return rate by region
group_means = compute_group_means(df, "region", "returned")
print("Return rate by region (sorted highest to lowest):")
print(group_means.round(4))


In [ ]:
# Test your code!
unittests.exercise_3(compute_group_means)


<a name='5---kde-overlays-by-target-class'></a>
## 5 - KDE Overlays by Target Class

A **Kernel Density Estimate (KDE)** shows the smooth probability distribution of a
continuous feature. Overlaying two KDEs — one for `returned=0` and one for
`returned=1` — lets you visually assess whether the feature's distribution **differs
between classes**:

- **Non-overlapping KDEs** → the feature strongly separates the two classes →
  high predictive power.
- **Fully overlapping KDEs** → the feature cannot distinguish classes alone →
  low predictive power.

Dashed vertical lines marking the **class means** make any location shift
immediately visible.

<a name='exercise-4---plot_kde_by_target'></a>
### **Exercise 4 - `plot_kde_by_target`**

**Your Task:**
Plot overlapping KDE curves for each class of the target variable, with vertical
dashed lines at each class mean.

**Requirements:**
- For each unique class in `target_col` (sorted), plot a KDE using
  `subset.plot.kde(ax=ax, ...)`.
- Label each curve `f"{target_col}={cls}"`.
- Use the colour list `['#4C72B0', '#C44E52']`, cycling by index.
- Add a dashed vertical line at the class mean with `alpha=0.8`.
- Set title to `f"{feature_col} KDE by {target_col}"` and xlabel to
  `feature_col`.
- Return the `Axes` object.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand)</font></b></summary>

  **Step-by-step guidance:**

  1. `classes = sorted(df[target_col].unique())`.
  2. `colors = ['#4C72B0', '#C44E52']`.
  3. For each `(i, cls)` in `enumerate(classes)`:
     - `subset = df.loc[df[target_col] == cls, feature_col].dropna()`
     - `subset.plot.kde(ax=ax, color=colors[i % len(colors)], label=f"{target_col}={cls}")`
     - `ax.axvline(subset.mean(), color=colors[i % len(colors)], linestyle='--',
       linewidth=1.2, alpha=0.8)`
  4. Set title, xlabel, call `ax.legend()`.

</details>


In [ ]:
# GRADED FUNCTION: plot_kde_by_target
def plot_kde_by_target(df, feature_col, target_col, figsize=(8, 4)):
    """
    Plot overlapping KDE curves for each class of the target variable.

    Args:
        df (pd.DataFrame): Input DataFrame.
        feature_col (str): Name of the numeric feature column.
        target_col (str): Name of the binary target column.
        figsize (tuple): Figure size. Default (8, 4).

    Returns:
        matplotlib.axes.Axes: The axes containing the KDE plot.
    """
    ### START CODE HERE ###
    # YOUR CODE HERE
    raise NotImplementedError("Implement this function")
    ### END CODE HERE ###


In [ ]:
# Demo: KDE overlays for order_value and customer_age
for feature in ["order_value", "customer_age"]:
    plot_kde_by_target(df, feature, "returned")
    plt.show()


In [ ]:
# Test your code!
unittests.exercise_4(plot_kde_by_target)


<a name='6---relationship-summary-table'></a>
## 6 - Relationship Summary Table

Combining correlation and group means into a single table lets you **prioritise
features at a glance** before model building. The summary answers three questions
for each numeric feature:

1. **How strong is the linear relationship?** → `correlation_with_target`
2. **What is the average feature value for non-returners vs returners?** →
   `mean_class_0`, `mean_class_1`
3. **How large is the gap between classes?** → `mean_diff`

A large `|mean_diff|` relative to the feature's scale indicates the feature may
be a strong predictor even when the linear correlation alone is modest.

<a name='exercise-5---build_relationship_summary'></a>
### **Exercise 5 - `build_relationship_summary`**

**Your Task:**
Build a summary DataFrame that combines Pearson correlation and class-level means
for each numeric feature.

**Requirements:**
- For each column in `numeric_cols`, compute:
  - `correlation_with_target` (Pearson correlation with `target_col`, rounded 4 dp)
  - `mean_class_0` (mean of feature where `target_col == 0`, rounded 2 dp)
  - `mean_class_1` (mean of feature where `target_col == 1`, rounded 2 dp)
  - `mean_diff` (`mean_class_1 - mean_class_0`, rounded 2 dp)
- Return a `pd.DataFrame` with these 4 columns, indexed by feature name.
- Sort rows by `|correlation_with_target|` descending.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand)</font></b></summary>

  **Step-by-step guidance:**

  1. Build `rows = {}`. For each `col` in `numeric_cols`, fill
     `rows[col] = {"correlation_with_target": ..., "mean_class_0": ..., ...}`.
  2. `result = pd.DataFrame(rows).T`.
  3. Add `result['abs_corr'] = result['correlation_with_target'].abs()`.
  4. `result.sort_values('abs_corr', ascending=False).drop(columns=['abs_corr'])`.

</details>


In [ ]:
# GRADED FUNCTION: build_relationship_summary
def build_relationship_summary(df, numeric_cols, categorical_col, target_col):
    """
    Build a summary DataFrame that helps prioritise numeric features.

    Args:
        df (pd.DataFrame): Input DataFrame.
        numeric_cols (list): List of numeric column names to summarise.
        categorical_col (str): Name of the categorical column (retained for API
                               consistency; not used in the numeric summary).
        target_col (str): Name of the binary target column.

    Returns:
        pd.DataFrame: DataFrame indexed by feature name with columns
                      'correlation_with_target', 'mean_class_0',
                      'mean_class_1', 'mean_diff'.
                      Sorted by |correlation_with_target| descending.
    """
    ### START CODE HERE ###
    # YOUR CODE HERE
    raise NotImplementedError("Implement this function")
    ### END CODE HERE ###


In [ ]:
# Demo: relationship summary for all 5 numeric features
numeric_cols = [
    "order_value", "days_to_ship", "items_ordered",
    "customer_age", "days_since_last_purchase"
]
summary = build_relationship_summary(df, numeric_cols, "region", "returned")
print("Feature-Target Relationship Summary (sorted by |correlation| descending):")
print(summary.to_string())
print()
print("Top feature by correlation:", summary.index[0])
print("Largest mean_diff :", summary['mean_diff'].abs().idxmax())


In [ ]:
# Test your code!
unittests.exercise_5(build_relationship_summary)


<a name='7---putting-it-all-together'></a>
## 7 - Putting It All Together

Excellent work! Let's combine all five analyses into a single dashboard to get
the full picture of feature–target relationships in one view.


In [ ]:
import matplotlib.gridspec as gridspec

numeric_cols = [
    "order_value", "days_to_ship", "items_ordered",
    "customer_age", "days_since_last_purchase"
]
target_col = "returned"

# Build all summaries
summary     = build_relationship_summary(df, numeric_cols, "region", target_col)
correlations = compute_feature_target_correlations(df, numeric_cols, target_col)
group_means  = compute_group_means(df, "region", target_col)

fig = plt.figure(figsize=(18, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.50, wspace=0.38)

# ── Top-left: correlation bar chart ──────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
sorted_corr  = correlations.sort_values()
colors_corr  = ['#4C72B0' if v >= 0 else '#C44E52' for v in sorted_corr.values]
ax1.barh(sorted_corr.index, sorted_corr.values, color=colors_corr, edgecolor='white')
ax1.axvline(0, color='black', linewidth=0.8)
ax1.set_title(f"Feature Correlations with {target_col}", fontsize=10)
ax1.set_xlabel("Pearson Correlation", fontsize=9)

# ── Top-middle: KDE for top feature ──────────────────────────────────────
top_feature = summary.index[0]
ax2 = fig.add_subplot(gs[0, 1])
classes    = sorted(df[target_col].unique())
colors_kde = ['#4C72B0', '#C44E52']
for i, cls in enumerate(classes):
    subset = df.loc[df[target_col] == cls, top_feature].dropna()
    subset.plot.kde(ax=ax2, color=colors_kde[i], label=f"{target_col}={cls}")
    ax2.axvline(subset.mean(), color=colors_kde[i], linestyle='--', linewidth=1.2, alpha=0.8)
ax2.set_title(f"{top_feature} KDE by {target_col}", fontsize=10)
ax2.set_xlabel(top_feature, fontsize=9)
ax2.legend(fontsize=8)

# ── Top-right: KDE for second feature ────────────────────────────────────
second_feature = summary.index[1] if len(summary) > 1 else numeric_cols[1]
ax3 = fig.add_subplot(gs[0, 2])
for i, cls in enumerate(classes):
    subset = df.loc[df[target_col] == cls, second_feature].dropna()
    subset.plot.kde(ax=ax3, color=colors_kde[i], label=f"{target_col}={cls}")
    ax3.axvline(subset.mean(), color=colors_kde[i], linestyle='--', linewidth=1.2, alpha=0.8)
ax3.set_title(f"{second_feature} KDE by {target_col}", fontsize=10)
ax3.set_xlabel(second_feature, fontsize=9)
ax3.legend(fontsize=8)

# ── Bottom-left: group means bar chart ────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 0])
bar_colors = ['#C44E52' if v == group_means.max() else '#4C72B0' for v in group_means.values]
ax4.bar(group_means.index, group_means.values, color=bar_colors, edgecolor='white')
ax4.set_title("Return Rate by Region", fontsize=10)
ax4.set_xlabel("Region", fontsize=9)
ax4.set_ylabel("Mean Return Rate", fontsize=9)
ax4.tick_params(axis='x', labelsize=8)

# ── Bottom-right: relationship summary table ──────────────────────────────
ax5 = fig.add_subplot(gs[1, 1:])
ax5.axis('off')
col_labels = ['Feature', 'Corr w/ Target', 'Mean (0)', 'Mean (1)', 'Mean Diff']
table_data = [
    [idx,
     f"{row['correlation_with_target']:.4f}",
     f"{row['mean_class_0']:.2f}",
     f"{row['mean_class_1']:.2f}",
     f"{row['mean_diff']:.2f}"]
    for idx, row in summary.iterrows()
]
tbl = ax5.table(cellText=table_data, colLabels=col_labels,
                cellLoc='center', loc='center', bbox=[0, 0.05, 1, 0.9])
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
ax5.set_title("Relationship Summary Table", fontsize=10, pad=12)

plt.suptitle("Feature-Target Relationship Dashboard",
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
numeric_cols = [
    "order_value", "days_to_ship", "items_ordered",
    "customer_age", "days_since_last_purchase"
]
summary      = build_relationship_summary(df, numeric_cols, "region", "returned")
correlations = compute_feature_target_correlations(df, numeric_cols, "returned")
group_means  = compute_group_means(df, "region", "returned")

top_corr_feature    = correlations.index[0]
top_corr_value      = correlations.iloc[0]
highest_risk_region = group_means.index[0]
highest_risk_rate   = group_means.iloc[0]
largest_diff_feat   = summary['mean_diff'].abs().idxmax()

print("=" * 60)
print("  FEATURE-TARGET RELATIONSHIP ANALYSIS SUMMARY")
print("=" * 60)
print(f"\n  Dataset: {len(df)} orders, {int(df['returned'].sum())} returned "
      f"({df['returned'].mean() * 100:.1f}% return rate)")
print(f"\n  Top correlated feature : '{top_corr_feature}' (r = {top_corr_value:+.4f})")
print(f"  Largest class mean gap : '{largest_diff_feat}'")
print(f"    mean_diff = {summary.loc[largest_diff_feat, 'mean_diff']:.2f}")
print(f"\n  Highest-risk region    : '{highest_risk_region}' "
      f"(return rate = {highest_risk_rate:.1%})")
print(f"\n  All feature correlations with 'returned':")
for feat, corr in correlations.items():
    bar  = chr(9608) * max(1, int(abs(corr) * 80))
    sign = '+' if corr >= 0 else '-'
    print(f"    {feat:36s} {sign}{abs(corr):.4f}  {bar}")
print("\n  Recommended next steps:")
print("  1. Prioritise features with |r| > 0.05 for model building.")
print("  2. Include 'region' as a categorical feature (notable rate differences).")
print("  3. Consider interaction terms between top correlated features.")
print("=" * 60)
print("\n\033[92m  Congratulations on completing the Feature-Target Relationship\n"
      "  Mapping assignment! You have built a complete EDA pipeline.\033[0m")
